In [1]:
import numpy as np
import math
from pathlib import Path
import torch
import json
import trimesh

In [2]:
smplx_path = Path("data/THuman_2.0_smplx_paras/0001/mesh_smplx.obj")
raw_path = Path("data/THuman_2.0/0001/0001.obj")

mesh_smplx = trimesh.load(smplx_path)
mesh_raw = trimesh.load(raw_path)

In [3]:
camera_front = json.load(open("/Users/lemon/Documents/TUD/Thesis/avatar-benchmark/data/THuman_cameras/thuman_front.json"))
camera_back = json.load(open("/Users/lemon/Documents/TUD/Thesis/avatar-benchmark/data/THuman_cameras/thuman_back.json"))
camera_left = json.load(open("/Users/lemon/Documents/TUD/Thesis/avatar-benchmark/data/THuman_cameras/thuman_left.json"))
camera_right = json.load(open("/Users/lemon/Documents/TUD/Thesis/avatar-benchmark/data/THuman_cameras/thuman_right.json"))

camera_list = [camera_front, camera_back, camera_left, camera_right]

def load_camera_params(camera_json, coords):
    """
    Json format:
        "K": K.tolist(),
        "viewmat": w2c.tolist(),
        "coords": "+z",
        "type": "w2c",
        "c2w_pyrender": c2w_pyr.tolist(),
        "image_size": [int(W), int(H)],
        "yfov_deg": float(yfov_deg),
    """
    
    K = np.array(camera_json["K"], dtype=float)
    w2c = np.array(camera_json["viewmat"], dtype=float)
    forward = camera_json["coords"]
    c2w_pyrender = np.array(camera_json["c2w_pyrender"], dtype=float)
    image_size = camera_json["image_size"]
    yfov_deg = camera_json["yfov_deg"]
    
    # If the camera is in OpenGL convention, we need to convert it to CV convention
    if coords == "-z":
        # OpenGL: camera looks along -Z, so +Z points backward
        # We need to flip the Z-axis to convert to CV convention
        w2c[:3, 2] = -w2c[:3, 2]
    elif coords == "+z":
        # CV: camera looks along +Z, so +Z points forward
        # No change needed
        pass
    return K, w2c

In [4]:
import pyrender

def render_to_2d_plane(mesh_, K, c2w, image_size):
    """
    Render the 3D mesh to a 2D plane using the camera parameters.
    
    Args:
        mesh_: trimesh mesh object
        K: 3x3 intrinsics matrix
        c2w: 4x4 camera-to-world pose matrix (pyrender convention)
        image_size: [width, height]
    """
    # Create a pyrender scene
    scene = pyrender.Scene()
    # Add the mesh to the scene
    mesh_node = pyrender.Mesh.from_trimesh(mesh_)
    scene.add(mesh_node)
    # Create a camera using intrinsics
    camera = pyrender.IntrinsicsCamera(fx=K[0, 0], fy=K[1, 1], cx=K[0, 2], cy=K[1, 2])
    # Add the camera to the scene with camera-to-world pose
    scene.add(camera, pose=c2w)
    # Create a renderer
    renderer = pyrender.OffscreenRenderer(viewport_width=image_size[0], viewport_height=image_size[1])
    # Render the scene
    color, depth = renderer.render(scene)
    renderer.delete()
    return color, depth

In [5]:
for i, camera_json in enumerate(camera_list):
    # Load the camera-to-world pose for pyrender (OpenGL convention)
    K = np.array(camera_json["K"], dtype=float)
    c2w_pyrender = np.array(camera_json["c2w_pyrender"], dtype=float)
    image_size = camera_json["image_size"]
    
    print(f"Rendering view {i} with camera pose at: {c2w_pyrender[:3, 3]}")
    color, _ = render_to_2d_plane(mesh_smplx, K, c2w_pyrender, image_size)
    
    import cv2
    cv2.imwrite(f"rendered_{i}.png", color)
    print(f"Saved rendered_{i}.png")

Rendering view 0 with camera pose at: [0.  0.  1.5]
Saved rendered_0.png
Rendering view 1 with camera pose at: [ 0.   0.  -1.5]
Saved rendered_1.png
Rendering view 2 with camera pose at: [-1.5  0.   0. ]
Saved rendered_0.png
Rendering view 1 with camera pose at: [ 0.   0.  -1.5]
Saved rendered_1.png
Rendering view 2 with camera pose at: [-1.5  0.   0. ]
Saved rendered_2.png
Rendering view 3 with camera pose at: [1.5 0.  0. ]
Saved rendered_3.png
Saved rendered_2.png
Rendering view 3 with camera pose at: [1.5 0.  0. ]
Saved rendered_3.png


In [6]:
from src.avatar_utils.smplx_loader import load_smplx_vertices

for i, camera_json in enumerate(camera_list):
    # Load the camera matrices
    K = np.array(camera_json["K"], dtype=float)
    w2c = np.array(camera_json["viewmat"], dtype=float)
    image_size = camera_json["image_size"]

    print(f"\n=== Processing view {i} ===")
    
    # The w2c matrix from gsplat already has the correct convention built-in
    # No need to pass coords parameter - it's handled automatically
    vertices, vertices_2d = load_smplx_vertices(smplx_path, K, w2c)
    
    # Choose random vertices to visualize
    num_vertices = vertices.shape[0]
    random_indices = np.random.choice(num_vertices, size=4000, replace=False)
    vertices_2d_sampled = vertices_2d[random_indices]
    
    # Visualize the 2D projections
    import cv2
    canvas = np.zeros((image_size[1], image_size[0], 3), dtype=np.uint8)
    for v in vertices_2d_sampled:
        x, y = int(v[0]), int(v[1])
        if 0 <= x < image_size[0] and 0 <= y < image_size[1]:
            cv2.circle(canvas, (x, y), radius=3, color=(0, 255, 0), thickness=-1)
    cv2.imwrite(f"vertices_2d_{i}.png", canvas)
    print(f"Saved vertices_2d_{i}.png")



=== Processing view 0 ===
Saved vertices_2d_0.png

=== Processing view 1 ===
Saved vertices_2d_1.png

=== Processing view 2 ===
Saved vertices_2d_2.png

=== Processing view 3 ===
Saved vertices_2d_3.png
Saved vertices_2d_0.png

=== Processing view 1 ===
Saved vertices_2d_1.png

=== Processing view 2 ===
Saved vertices_2d_2.png

=== Processing view 3 ===
Saved vertices_2d_3.png
